# Floorplan → 3D: multi-task segmentation (EfficientNet-B3 U-Net, 768 px, 2×T4)

**Predicts:** rooms (incl. walls/railings) · icons (doors, windows, **stairs**, fixtures) · wall-junction / opening heatmaps for vectorising into 3D.
**Data:** CubiCasa5K (auto-downloaded; licence CC BY-NC-SA 4.0 — non-commercial).

### Before running (one time)
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On** (phone may need *Verify phone* on your Kaggle account)
3. **Add-ons → Secrets → Add** `HF_TOKEN` = your Hugging Face **write** token, and tick it for this notebook
4. **Save Version → Save & Run All (Commit)** — runs in the background, so you can close the phone browser.

### While it runs / after
* Progress, metrics and checkpoints appear in your private HF repo (`metrics.json`, `status.json`, `checkpoints/`).
* Kaggle caps a run at 12 h; training sizes its schedule to fit and stops cleanly before that.
* To train longer: run the same notebook again — it resumes from `checkpoints/last.pt` (raise `EXTRA_EPOCHS`).


In [ ]:
# ===================== CONFIG (edit here only) =====================
import os, time, json, sys, subprocess, shutil, re
NOTEBOOK_START = time.time()

MODEL_REPO_NAME   = "floorplan-multitask-effb3-768"   # private HF model repo (checkpoints)
DATASET_REPO_NAME = "cubicasa5k-floorplan-cache"      # private HF dataset repo (preprocessed data)
TIME_BUDGET_HOURS = 11.25     # Kaggle kills sessions at 12 h; leaves a safety buffer
MAX_EPOCHS        = 150       # upper bound; schedule auto-shrinks to fit the time budget
EXTRA_EPOCHS      = 0         # on a re-run: >0 extends a finished schedule by this many epochs

ENCODER   = "tu-efficientnet_b3"
CROP      = 768               # training crop size
MAX_SIDE  = 1536              # plans are cached with longest side <= this (keeps thin walls)
SIGMA     = 3.0               # junction heatmap gaussian (px)
LR        = 1e-3              # decoder/head LR; encoder uses LR * 0.5
SEED      = 42

SRC_DIR   = "/kaggle/working/fp_src"
CKPT_DIR  = "/kaggle/working/checkpoints"
BIG_DIR   = "/tmp/fp"            # large scratch (not saved as notebook output)
CACHE_DIR = f"{BIG_DIR}/cache"
os.makedirs(SRC_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(BIG_DIR, exist_ok=True)
print("config ok")

In [ ]:
!pip install -q "segmentation-models-pytorch==0.6.0" "timm==1.0.31" "huggingface_hub>=0.25" svgpathtools 2>&1 | grep -v "already satisfied" | tail -3
!pip install -q hf_xet 2>&1 | tail -1 || true

In [ ]:
%%writefile /kaggle/working/fp_src/fp_common.py
"""Shared code: classes, cached dataset + augmentation, model, losses, metrics."""
import json
import math
import os
import random

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

cv2.setNumThreads(0)

# ----------------------------------------------------------------------------- classes
# Room head (from the official CubiCasa5K "rooms_selected" mapping)
ROOM_CLASSES = ["Background", "Outdoor", "Wall", "Kitchen", "LivingRoom", "Bedroom",
                "Bath", "Entry/Hallway", "Railing", "Storage", "Garage", "OtherRoom"]
# Icon head (official "icons_selected" mapping) + Stairs (parsed separately, id 11)
ICON_CLASSES = ["NoIcon", "Window", "Door", "Closet", "ElectricalAppliance", "Toilet",
                "Sink", "SaunaBench", "Fireplace", "Bathtub", "Chimney", "Stairs"]
# Junction heatmaps (orientation-free so flips/rotations stay valid)
HEAT_CLASSES = ["WallJunction", "OpeningEndpoint", "IconCorner"]

N_ROOM, N_ICON, N_HEAT = len(ROOM_CLASSES), len(ICON_CLASSES), len(HEAT_CLASSES)
N_OUT = N_ROOM + N_ICON + N_HEAT
IGNORE = 255
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32) * 255.0
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32) * 255.0


# ----------------------------------------------------------------------------- heatmaps
def render_heatmaps(points, h, w, sigma):
    """points: (N,3) float array [x, y, type]. Returns (N_HEAT, h, w) float32, peak exactly 1."""
    heat = np.zeros((N_HEAT, h, w), dtype=np.float32)
    if points is None or len(points) == 0:
        return heat
    r = int(math.ceil(3 * sigma))
    ax = np.arange(-r, r + 1, dtype=np.float32)
    g = np.exp(-(ax[None, :] ** 2 + ax[:, None] ** 2) / (2 * sigma * sigma)).astype(np.float32)
    for x, y, t in points:
        cx, cy, t = int(round(float(x))), int(round(float(y))), int(t)
        if not (0 <= cx < w and 0 <= cy < h) or not (0 <= t < N_HEAT):
            continue
        x0, x1 = max(0, cx - r), min(w, cx + r + 1)
        y0, y1 = max(0, cy - r), min(h, cy + r + 1)
        patch = g[y0 - (cy - r):y1 - (cy - r), x0 - (cx - r):x1 - (cx - r)]
        np.maximum(heat[t, y0:y1, x0:x1], patch, out=heat[t, y0:y1, x0:x1])
    return heat


# ----------------------------------------------------------------------------- dataset
class CachedFloorplans(torch.utils.data.Dataset):
    """Reads the preprocessed cache: <root>/data/<id>_img.png, _lbl.png (B=room, G=icon), _pts.npy"""

    def __init__(self, root, split, train, crop=768, sigma=3.0, scale_range=(0.75, 1.3)):
        with open(os.path.join(root, "index.json")) as f:
            index = json.load(f)
        self.items = index[split]
        self.root = os.path.join(root, "data")
        self.train, self.crop, self.sigma, self.scale_range = train, crop, sigma, scale_range

    def __len__(self):
        return len(self.items)

    def _load(self, i):
        sid = self.items[i]["id"]
        img = cv2.imread(os.path.join(self.root, sid + "_img.png"), cv2.IMREAD_COLOR)
        lbl = cv2.imread(os.path.join(self.root, sid + "_lbl.png"), cv2.IMREAD_UNCHANGED)
        if img is None or lbl is None:
            raise RuntimeError(f"Corrupt cache entry {sid}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pts = np.load(os.path.join(self.root, sid + "_pts.npy")).astype(np.float32).reshape(-1, 3)
        return img, lbl[:, :, 0].copy(), lbl[:, :, 1].copy(), pts

    def __getitem__(self, i):
        img, room, icon, pts = self._load(i)
        if self.train:
            img, room, icon, pts = self._augment(img, room, icon, pts)
        else:
            img, room, icon = self._pad32(img, room, icon)
        h, w = room.shape
        heat = render_heatmaps(pts, h, w, self.sigma)
        x = (img.astype(np.float32) - MEAN) / STD
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1))),
                torch.from_numpy(np.ascontiguousarray(room)).long(),
                torch.from_numpy(np.ascontiguousarray(icon)).long(),
                torch.from_numpy(heat))

    @staticmethod
    def _pad32(img, room, icon):
        h, w = room.shape
        ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
        if ph or pw:
            img = cv2.copyMakeBorder(img, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=(255, 255, 255))
            room = cv2.copyMakeBorder(room, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=IGNORE)
            icon = cv2.copyMakeBorder(icon, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=IGNORE)
        return img, room, icon

    def _augment(self, img, room, icon, pts):
        S = self.crop
        h, w = room.shape
        lo, hi = self.scale_range
        s = math.exp(random.uniform(math.log(lo), math.log(hi)))
        src = S / s  # crop size in source pixels
        x0 = random.uniform(0, w - src) if w > src else -random.uniform(0, src - w)
        y0 = random.uniform(0, h - src) if h > src else -random.uniform(0, src - h)
        M = np.array([[s, 0, -x0 * s], [0, s, -y0 * s]], dtype=np.float32)
        interp = cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR
        img = cv2.warpAffine(img, M, (S, S), flags=interp, borderMode=cv2.BORDER_CONSTANT,
                             borderValue=(255, 255, 255))
        room = cv2.warpAffine(room, M, (S, S), flags=cv2.INTER_NEAREST,
                              borderMode=cv2.BORDER_CONSTANT, borderValue=IGNORE)
        icon = cv2.warpAffine(icon, M, (S, S), flags=cv2.INTER_NEAREST,
                              borderMode=cv2.BORDER_CONSTANT, borderValue=IGNORE)
        if len(pts):
            pts = pts.copy()
            pts[:, 0] = (pts[:, 0] - x0) * s
            pts[:, 1] = (pts[:, 1] - y0) * s
        # rotations / flips (all heads are orientation-free, so labels stay valid)
        k = random.randint(0, 3)
        for _ in range(k):  # 90 deg counter-clockwise
            img, room, icon = np.rot90(img), np.rot90(room), np.rot90(icon)
            if len(pts):
                pts[:, 0], pts[:, 1] = pts[:, 1].copy(), (S - 1) - pts[:, 0]
        if random.random() < 0.5:
            img, room, icon = img[:, ::-1], room[:, ::-1], icon[:, ::-1]
            if len(pts):
                pts[:, 0] = (S - 1) - pts[:, 0]
        img = np.ascontiguousarray(img).astype(np.float32)
        # photometric
        if random.random() < 0.3:
            g = img.mean(axis=2, keepdims=True)
            img = np.repeat(g, 3, axis=2)
        if random.random() < 0.8:
            img = (img - 128.0) * random.uniform(0.75, 1.25) + 128.0 + random.uniform(-25, 25)
        if random.random() < 0.2:
            img = cv2.GaussianBlur(img, (0, 0), random.uniform(0.3, 1.0))
        if random.random() < 0.25:
            img = img + np.random.normal(0, random.uniform(2, 8), img.shape).astype(np.float32)
        img = np.clip(img, 0, 255).astype(np.uint8)
        return img, np.ascontiguousarray(room), np.ascontiguousarray(icon), pts


# ----------------------------------------------------------------------------- model
def build_model(encoder="tu-efficientnet_b3", weights="imagenet"):
    import segmentation_models_pytorch as smp
    model = smp.Unet(encoder_name=encoder, encoder_weights=weights, classes=N_OUT,
                     decoder_attention_type="scse")
    # focal-loss prior for heatmap logits (CenterNet init)
    head = [m for m in model.segmentation_head.modules() if isinstance(m, nn.Conv2d)][-1]
    with torch.no_grad():
        head.bias[N_ROOM + N_ICON:].fill_(-2.19)
    # Some smp decoder blocks own parameters that never get gradients (e.g. the attention on
    # the missing skip of the last block). DDP aborts on those, so detect + freeze them.
    saved = {k: v.clone() for k, v in model.state_dict().items()}  # keep BN stats untouched
    model.train()
    model(torch.randn(2, 3, 64, 64)).float().sum().backward()
    for p in model.parameters():
        if p.grad is None:
            p.requires_grad_(False)
        p.grad = None
    model.load_state_dict(saved)
    return model


def split_logits(out):
    return out[:, :N_ROOM], out[:, N_ROOM:N_ROOM + N_ICON], out[:, N_ROOM + N_ICON:]


# ----------------------------------------------------------------------------- losses
def dice_loss(logits, target, n_cls, ignore=IGNORE, eps=1.0):
    valid = (target != ignore)
    t = torch.where(valid, target, torch.zeros_like(target))
    prob = logits.softmax(1) * valid.unsqueeze(1)
    oh = F.one_hot(t, n_cls).permute(0, 3, 1, 2).float() * valid.unsqueeze(1)
    dims = (0, 2, 3)
    inter = (prob * oh).sum(dims)
    card = prob.sum(dims) + oh.sum(dims)
    present = oh.sum(dims) > 0  # only classes present in the batch
    dice = (2 * inter + eps) / (card + eps)
    if present.any():
        return 1 - dice[present].mean()
    return logits.sum() * 0.0


def focal_ce(logits, target, gamma=2.0, ignore=IGNORE):
    ce = F.cross_entropy(logits, target, ignore_index=ignore, reduction="none")
    valid = (target != ignore)
    if not valid.any():
        return logits.sum() * 0.0
    pt = torch.exp(-ce)
    return (((1 - pt) ** gamma) * ce)[valid].mean()


def heat_focal(logits, gt, alpha=2.0, beta=4.0):
    pred = torch.sigmoid(logits).clamp(1e-4, 1 - 1e-4)
    pos = gt.ge(0.999).float()
    neg = 1 - pos
    pos_loss = torch.log(pred) * (1 - pred) ** alpha * pos
    neg_loss = torch.log(1 - pred) * pred ** alpha * (1 - gt) ** beta * neg
    n_pos = pos.sum().clamp(min=1.0)
    return -(pos_loss.sum() + neg_loss.sum()) / n_pos


class MultiTaskLoss(nn.Module):
    def __init__(self, w_room=1.0, w_icon=1.0, w_heat=1.0):
        super().__init__()
        self.w = (w_room, w_icon, w_heat)

    def forward(self, out, room, icon, heat):
        r, i, h = [t.float() for t in split_logits(out)]
        l_room = F.cross_entropy(r, room, ignore_index=IGNORE) + dice_loss(r, room, N_ROOM)
        l_icon = focal_ce(i, icon) + dice_loss(i, icon, N_ICON)
        l_heat = heat_focal(h, heat)
        total = self.w[0] * l_room + self.w[1] * l_icon + self.w[2] * l_heat
        return total, {"room": l_room.detach(), "icon": l_icon.detach(), "heat": l_heat.detach()}


# ----------------------------------------------------------------------------- metrics
def update_confusion(cm, pred, target, n_cls, ignore=IGNORE):
    valid = target != ignore
    idx = target[valid] * n_cls + pred[valid]
    cm += torch.bincount(idx, minlength=n_cls * n_cls).reshape(n_cls, n_cls)
    return cm


def iou_from_confusion(cm):
    cm = cm.double()
    tp = cm.diag()
    denom = cm.sum(0) + cm.sum(1) - tp
    iou = torch.where(denom > 0, tp / denom.clamp(min=1), torch.full_like(tp, float("nan")))
    return iou  # nan for classes absent from both pred and gt


class EMA:
    """Exponential moving average of weights (fp32), with warmup-aware decay."""

    def __init__(self, model, decay=0.999):
        import copy
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.updates = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


In [ ]:
%%writefile /kaggle/working/fp_src/fp_prep.py
"""Data acquisition + one-time preprocessing of CubiCasa5K into a fast training cache.

Order of preference:
  1. cache tar already in your private HF dataset repo  -> download + untar (fast resume)
  2. CubiCasa5K attached as a Kaggle dataset            -> parse from /kaggle/input
  3. official Zenodo zip (5.5 GB, md5-checked)           -> download, parse
Then the cache is uploaded to the HF dataset repo so later runs skip steps 2/3.
"""
import glob
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
import traceback
import zipfile
from multiprocessing import Pool

import cv2
import numpy as np

ZENODO_URL = "https://zenodo.org/records/2613548/files/cubicasa5k.zip?download=1"
ZENODO_MD5 = "0ce0b203d1e3c125b51087b219bd23b9"
CACHE_VERSION = "v1"

_PARSER_DIR = None


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


# ----------------------------------------------------------------------------- raw data
def find_raw_root(search_roots=("/kaggle/input",)):
    for base in search_roots:
        if not os.path.isdir(base):
            continue
        for txt in glob.glob(os.path.join(base, "**", "train.txt"), recursive=True):
            root = os.path.dirname(txt)
            if (os.path.exists(os.path.join(root, "val.txt"))
                    and glob.glob(os.path.join(root, "high_quality*"))):
                return root
    return None


def md5sum(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def _download(url, dst, label):
    """Resumable HTTP download that prints a progress line every ~2% (log-friendly)."""
    import urllib.request
    have = os.path.getsize(dst) if os.path.exists(dst) else 0
    req = urllib.request.Request(url, headers={"User-Agent": "floorplan-trainer",
                                               **({"Range": f"bytes={have}-"} if have else {})})
    with urllib.request.urlopen(req, timeout=120) as r:
        if have and r.status != 206:  # server ignored resume -> start over
            have = 0
        total = have + int(r.headers.get("Content-Length") or 0)
        t0, done, next_pct = time.time(), have, 0
        with open(dst, "ab" if have else "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
                done += len(chunk)
                pct = 100 * done / total if total else 0
                if pct >= next_pct or done == total:
                    sp = (done - have) / max(time.time() - t0, 1e-6)
                    eta = (total - done) / sp if sp > 0 else 0
                    bar = "#" * int(pct / 5) + "-" * (20 - int(pct / 5))
                    log(f"{label} [{bar}] {pct:5.1f}%  {done / 1e9:.2f}/{total / 1e9:.2f} GB  "
                        f"{sp / 1e6:.1f} MB/s  ETA {int(eta // 60)}m{int(eta % 60):02d}s")
                    next_pct = int(pct) + 2
    if total and os.path.getsize(dst) < total:
        raise IOError("download incomplete")


def download_zenodo(work_dir, retries=6):
    os.makedirs(work_dir, exist_ok=True)
    zpath = os.path.join(work_dir, "cubicasa5k.zip")
    for attempt in range(1, retries + 1):
        log(f"Downloading CubiCasa5K from Zenodo, 5.5 GB (attempt {attempt}/{retries}) ...")
        try:
            _download(ZENODO_URL, zpath, "download")
            log("Download finished, verifying md5 (~1 min) ...")
            if md5sum(zpath) == ZENODO_MD5:
                log("md5 OK")
                return zpath
            log("md5 mismatch -> deleting and retrying")
            os.remove(zpath)
        except Exception as e:
            log(f"Download interrupted ({type(e).__name__}: {e}); resuming ...")
        time.sleep(min(60, 10 * attempt))
    raise RuntimeError("Could not download CubiCasa5K from Zenodo. Attach the dataset on Kaggle "
                       "(Add Data -> search 'cubicasa5k') and rerun.")


def extract_needed(zpath, out_dir):
    """Extract only split files, F1_scaled.png and model.svg (skips the big originals)."""
    log("Extracting needed files ...")
    with zipfile.ZipFile(zpath) as z:
        names = [n for n in z.namelist()
                 if n.endswith((".txt", "F1_scaled.png", "model.svg"))]
        for k, n in enumerate(names, 1):
            z.extract(n, out_dir)
            if k % 1000 == 0 or k == len(names):
                log(f"  extracted {k}/{len(names)} files ({100 * k / len(names):.0f}%)")
    root = find_raw_root([out_dir])
    if root is None:
        raise RuntimeError("Extracted archive has unexpected layout (train.txt not found).")
    log(f"Extracted {len(names)} files, root = {root}")
    return root


# ----------------------------------------------------------------------------- parser
def setup_parser(parser_dir):
    """Vendor the official CubiCasa5K SVG parser (house.py + svg_utils.py) without its
    lmdb/torch package imports."""
    if os.path.exists(os.path.join(parser_dir, "house.py")):
        return parser_dir
    tmp = parser_dir + "_repo"
    if not os.path.exists(tmp):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/CubiCasa/CubiCasa5k.git", tmp], check=True)
    os.makedirs(parser_dir, exist_ok=True)
    for fn in ("house.py", "svg_utils.py"):
        src = open(os.path.join(tmp, "floortrans", "loaders", fn)).read()
        src = src.replace("from floortrans.loaders.svg_utils import", "from svg_utils import")
        src = src.replace("is 'H'", "== 'H'").replace("is 'V'", "== 'V'")
        open(os.path.join(parser_dir, fn), "w").write(src)
    return parser_dir


def _init_worker(parser_dir):
    global _PARSER_DIR
    _PARSER_DIR = parser_dir
    if parser_dir not in sys.path:
        sys.path.insert(0, parser_dir)
    cv2.setNumThreads(1)
    import warnings, logging
    warnings.filterwarnings("ignore")
    logging.getLogger().setLevel(logging.ERROR)


def _parse_stairs(svg_path, shape, icons):
    """Stairs are commented out in the official parser; we paint Flight/Winding polygons
    as icon class 11 where no other icon is present. Returns number of stair polygons."""
    from xml.dom import minidom
    from svg_utils import get_polygon
    h, w = shape
    n = 0
    dom = minidom.parse(svg_path)
    for g in dom.getElementsByTagName("g"):
        if "Stairs" not in g.getAttribute("class").split(" "):
            continue
        parts = [c for c in g.childNodes if c.nodeType == c.ELEMENT_NODE
                 and c.getAttribute("class") in ("Flight", "Winding")] or [g]
        for c in parts:
            try:
                rr, cc = get_polygon(c)
            except Exception:
                continue
            keep = (rr >= 0) & (rr < h) & (cc >= 0) & (cc < w)
            rr, cc = rr[keep], cc[keep]
            if len(rr) == 0:
                continue
            sel = icons[rr, cc] == 0
            icons[rr[sel], cc[sel]] = 11
            n += 1
    return n


def _heat_group(ch):
    return 0 if ch <= 12 else (1 if ch <= 16 else 2)


def process_one(args):
    raw_root, folder, out_dir, max_side = args
    sid = folder.strip("/").replace("/", "__")
    try:
        import house as cc_house
        d = os.path.join(raw_root, folder.strip("/"))
        img = cv2.imread(os.path.join(d, "F1_scaled.png"), cv2.IMREAD_UNCHANGED)
        if img is None:
            return {"id": sid, "error": "missing image"}
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        elif img.shape[2] == 4:  # composite transparency on white
            a = img[:, :, 3:4].astype(np.float32) / 255.0
            img = (img[:, :, :3].astype(np.float32) * a + 255.0 * (1 - a)).astype(np.uint8)
        h, w = img.shape[:2]
        house = cc_house.House(os.path.join(d, "model.svg"), h, w)
        room = house.walls.astype(np.uint8)
        icon = house.icons.astype(np.uint8)
        n_stairs = _parse_stairs(os.path.join(d, "model.svg"), (h, w), icon)
        pts = []
        for ch, plist in house.get_heatmap_dict().items():
            g = _heat_group(int(ch))
            pts += [(float(x), float(y), g) for x, y in plist]
        pts = np.array(pts, dtype=np.float32).reshape(-1, 3)

        s = min(1.0, max_side / max(h, w))
        if s < 1.0:
            nw, nh = int(round(w * s)), int(round(h * s))
            img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
            room = cv2.resize(room, (nw, nh), interpolation=cv2.INTER_NEAREST)
            icon = cv2.resize(icon, (nw, nh), interpolation=cv2.INTER_NEAREST)
            pts[:, :2] *= s
            h, w = nh, nw
        cv2.imwrite(os.path.join(out_dir, sid + "_img.png"), img)  # BGR on disk
        lbl = np.dstack([room, icon, np.zeros_like(room)])
        cv2.imwrite(os.path.join(out_dir, sid + "_lbl.png"), lbl)
        np.save(os.path.join(out_dir, sid + "_pts.npy"), pts)
        return {"id": sid, "h": h, "w": w, "n_pts": int(len(pts)), "stairs": n_stairs}
    except Exception as e:
        return {"id": sid, "error": f"{type(e).__name__}: {e}",
                "tb": traceback.format_exc(limit=3)}


def build_cache(raw_root, cache_root, parser_dir, max_side, workers=None):
    out_dir = os.path.join(cache_root, "data")
    os.makedirs(out_dir, exist_ok=True)
    workers = workers or os.cpu_count() or 2
    index, failures, stairs_total = {}, [], 0
    for split in ("train", "val", "test"):
        folders = [l.strip() for l in open(os.path.join(raw_root, f"{split}.txt")) if l.strip()]
        log(f"Parsing {split}: {len(folders)} floorplans with {workers} workers ...")
        t0, items = time.time(), []
        jobs = [(raw_root, f, out_dir, max_side) for f in folders]
        with Pool(workers, initializer=_init_worker, initargs=(parser_dir,)) as pool:
            for k, r in enumerate(pool.imap_unordered(process_one, jobs, chunksize=4), 1):
                if "error" in r:
                    failures.append(r)
                else:
                    items.append(r)
                    stairs_total += r["stairs"]
                if k % 100 == 0 or k == len(jobs):
                    el = time.time() - t0
                    log(f"  parsing {split}: {k}/{len(jobs)} ({100 * k / len(jobs):.0f}%)  "
                        f"{el / k:.2f}s/plan  ETA {el / k * (len(jobs) - k) / 60:.1f} min")
        items.sort(key=lambda r: r["id"])
        index[split] = items
    n_total = sum(len(v) for v in index.values()) + len(failures)
    log(f"Done. ok={n_total - len(failures)} failed={len(failures)} stair polygons={stairs_total}")
    for f in failures[:5]:
        log("  failure:", f["id"], f["error"])
    if len(failures) > 0.05 * n_total:
        raise RuntimeError(f"Too many parse failures ({len(failures)}/{n_total}); see log above.")
    if stairs_total == 0:
        log("WARNING: no stairs parsed; the Stairs class will stay empty.")
    index["meta"] = {"max_side": max_side, "version": CACHE_VERSION,
                     "failures": [f["id"] for f in failures], "stairs_polygons": stairs_total}
    with open(os.path.join(cache_root, "index.json"), "w") as f:
        json.dump(index, f)
    return index


# ----------------------------------------------------------------------------- HF cache
def cache_name(max_side):
    return f"cubicasa5k_cache_side{max_side}_{CACHE_VERSION}.tar"


def try_download_cache(api, repo, max_side, cache_root, token):
    from huggingface_hub import hf_hub_download
    name = cache_name(max_side)
    try:
        files = api.list_repo_files(repo, repo_type="dataset", token=token)
    except Exception as e:
        log(f"Could not list dataset repo ({e}); will build cache.")
        return False
    if name not in files:
        return False
    log(f"Found cached data on HF ({name}); downloading ...")
    for attempt in range(3):
        try:
            p = hf_hub_download(repo, name, repo_type="dataset", token=token,
                                local_dir=os.path.dirname(cache_root.rstrip("/")) + "/hf_dl")
            os.makedirs(cache_root, exist_ok=True)
            with tarfile.open(p) as t:
                t.extractall(cache_root)
            os.remove(p)
            ok = os.path.exists(os.path.join(cache_root, "index.json"))
            log("Cache restored." if ok else "Cache tar was incomplete.")
            return ok
        except Exception as e:
            log(f"Cache download failed ({e}); retry {attempt + 1}/3")
            time.sleep(15)
    return False


def upload_cache(api, repo, max_side, cache_root, token):
    name = cache_name(max_side)
    tar_path = os.path.join(os.path.dirname(cache_root.rstrip("/")), name)
    log("Packing cache for HF upload ...")
    with tarfile.open(tar_path, "w") as t:
        t.add(os.path.join(cache_root, "index.json"), arcname="index.json")
        t.add(os.path.join(cache_root, "data"), arcname="data")
    log(f"Uploading {os.path.getsize(tar_path) / 1e9:.2f} GB to {repo} (dataset) ...")
    for attempt in range(3):
        try:
            api.upload_file(path_or_fileobj=tar_path, path_in_repo=name, repo_id=repo,
                            repo_type="dataset", token=token,
                            commit_message=f"Preprocessed CubiCasa5K cache (side {max_side})")
            log("Cache uploaded.")
            break
        except Exception as e:
            log(f"Cache upload failed ({e}); retry {attempt + 1}/3")
            time.sleep(30)
    else:
        log("WARNING: cache upload failed; training continues (next run will rebuild).")
    os.remove(tar_path)


def prepare_data(cache_root, work_dir, parser_dir, max_side, api, dataset_repo, token):
    if os.path.exists(os.path.join(cache_root, "index.json")):
        log("Local cache already present.")
        return json.load(open(os.path.join(cache_root, "index.json")))
    if api is not None and try_download_cache(api, dataset_repo, max_side, cache_root, token):
        return json.load(open(os.path.join(cache_root, "index.json")))
    raw = find_raw_root()
    if raw:
        log(f"Using Kaggle-attached CubiCasa5K at {raw}")
    else:
        zpath = download_zenodo(work_dir)
        raw = extract_needed(zpath, os.path.join(work_dir, "raw"))
        os.remove(zpath)
    setup_parser(parser_dir)
    index = build_cache(raw, cache_root, parser_dir, max_side)
    if raw.startswith(work_dir):
        shutil.rmtree(os.path.join(work_dir, "raw"), ignore_errors=True)
    if api is not None:
        upload_cache(api, dataset_repo, max_side, cache_root, token)
    return index


In [ ]:
%%writefile /kaggle/working/fp_src/train.py
"""Multi-task floorplan training (rooms + icons/doors/windows/stairs + junction heatmaps).
Launch:  torchrun --standalone --nproc_per_node=2 train.py --config run_config.json
Probe :  python train.py --config run_config.json --probe   (finds the largest safe batch)
"""
import argparse
import datetime
import json
import math
import os
import random
import shutil
import threading
import time

import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn

from fp_common import (CachedFloorplans, MultiTaskLoss, EMA, build_model, split_logits,
                       update_confusion, iou_from_confusion, ROOM_CLASSES, ICON_CLASSES,
                       N_ROOM, N_ICON)


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


# ----------------------------------------------------------------------------- HF uploads
class Uploader:
    """Background, retrying uploads so training never waits on (or dies from) the network."""

    def __init__(self, repo, token):
        self.repo, self.token = repo, token
        self.thread = None
        self.staging = os.path.join(os.environ.get("TMPDIR", "/tmp"), "hf_staging")
        os.makedirs(self.staging, exist_ok=True)

    def _run(self, items, msg):
        from huggingface_hub import HfApi
        api = HfApi(token=self.token)
        for attempt in range(1, 6):
            try:
                from huggingface_hub import CommitOperationAdd
                ops = [CommitOperationAdd(path_in_repo=dst, path_or_fileobj=src) for src, dst in items]
                api.create_commit(repo_id=self.repo, operations=ops, commit_message=msg,
                                  repo_type="model")
                log(f"[hf] uploaded: {', '.join(d for _, d in items)}")
                return
            except Exception as e:
                log(f"[hf] upload attempt {attempt}/5 failed: {type(e).__name__}: {e}")
                time.sleep(min(120, 15 * attempt))
        log("[hf] WARNING: giving up on this upload; next checkpoint will retry.")

    def submit(self, files, msg):
        """files: list of (local_path, path_in_repo). Snapshots files before returning."""
        if not self.repo or not self.token:
            return
        self.wait()
        snap = []
        for i, (src, dst) in enumerate(files):
            if os.path.exists(src):
                tmp = os.path.join(self.staging, f"{i}_{os.path.basename(src)}")
                shutil.copyfile(src, tmp)
                snap.append((tmp, dst))
        if snap:
            self.thread = threading.Thread(target=self._run, args=(snap, msg), daemon=True)
            self.thread.start()

    def wait(self, timeout=None):
        if self.thread is not None:
            self.thread.join(timeout)
            if not self.thread.is_alive():
                self.thread = None


# ----------------------------------------------------------------------------- helpers
def seed_worker(_):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


def set_lr(opt, step, steps_per_epoch, total_epochs, warmup_epochs, min_ratio=0.01):
    warm = max(1, int(warmup_epochs * steps_per_epoch))
    total = max(warm + 1, total_epochs * steps_per_epoch)
    if step < warm:
        f = (step + 1) / warm
    else:
        p = min(1.0, (step - warm) / max(1, total - warm))
        f = min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * p))
    for g in opt.param_groups:
        g["lr"] = g["base_lr"] * f
    return f


def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def signature(cfg):
    return {k: cfg[k] for k in ("encoder", "crop", "max_side", "sigma")} | {
        "n_room": N_ROOM, "n_icon": N_ICON}


def make_optimizer(model, cfg):
    enc, rest = [], []
    for n, p in model.named_parameters():
        if p.requires_grad:
            (enc if n.startswith("encoder.") else rest).append(p)
    groups = [{"params": enc, "base_lr": cfg["lr"] * cfg["encoder_lr_mult"]},
              {"params": rest, "base_lr": cfg["lr"]}]
    for g in groups:
        g["lr"] = g["base_lr"]
    return torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])


# ----------------------------------------------------------------------------- probe
def probe(cfg):
    """Find the largest per-GPU batch that trains at the crop size with >=15% memory headroom."""
    dev = torch.device("cuda:0")
    total_mem = torch.cuda.get_device_properties(0).total_memory
    model = build_model(cfg["encoder"], None).to(dev).train()
    opt = make_optimizer(model, cfg)
    scaler = torch.amp.GradScaler("cuda")
    crit = MultiTaskLoss(*cfg["loss_weights"])
    best = None
    for bs in cfg["probe_batches"]:
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            for _ in range(3):
                x = torch.randn(bs, 3, cfg["crop"], cfg["crop"], device=dev)
                room = torch.randint(0, N_ROOM, (bs, cfg["crop"], cfg["crop"]), device=dev)
                icon = torch.randint(0, N_ICON, (bs, cfg["crop"], cfg["crop"]), device=dev)
                heat = torch.rand(bs, 3, cfg["crop"], cfg["crop"], device=dev)
                with torch.autocast("cuda", dtype=torch.float16):
                    out = model(x)
                loss, _ = crit(out, room, icon, heat)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
            peak = torch.cuda.max_memory_allocated()
            log(f"probe bs={bs}: peak {peak / 2 ** 30:.2f} GiB / {total_mem / 2 ** 30:.2f} GiB")
            if peak < 0.85 * total_mem:
                best = bs
                break
        except torch.cuda.OutOfMemoryError:
            log(f"probe bs={bs}: OOM")
        finally:
            opt.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    best = best or min(cfg["probe_batches"])
    print(f"PROBE_BS={best}", flush=True)


# ----------------------------------------------------------------------------- validation
@torch.no_grad()
def validate(net, ds, rank, world, dev):
    net.eval()
    bench = torch.backends.cudnn.benchmark
    torch.backends.cudnn.benchmark = False  # every val plan has its own size
    cm_r = torch.zeros(N_ROOM, N_ROOM, dtype=torch.long, device=dev)
    cm_i = torch.zeros(N_ICON, N_ICON, dtype=torch.long, device=dev)
    idx = list(range(rank, len(ds), world))
    loader = torch.utils.data.DataLoader(torch.utils.data.Subset(ds, idx), batch_size=1,
                                         shuffle=False, num_workers=2,
                                         pin_memory=dev.type == "cuda")
    for x, room, icon, _ in loader:
        x, room, icon = x.to(dev, non_blocking=True), room.to(dev), icon.to(dev)
        with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
            out = net(x)
        r, i, _h = split_logits(out.float())
        update_confusion(cm_r, r.argmax(1), room, N_ROOM)
        update_confusion(cm_i, i.argmax(1), icon, N_ICON)
    if world > 1:
        dist.all_reduce(cm_r)
        dist.all_reduce(cm_i)
    torch.backends.cudnn.benchmark = bench
    iou_r, iou_i = iou_from_confusion(cm_r).cpu(), iou_from_confusion(cm_i).cpu()
    acc_r = (cm_r.diag().sum() / cm_r.sum().clamp(min=1)).item()
    miou_r = float(np.nanmean(iou_r.numpy()))
    miou_i = float(np.nanmean(iou_i.numpy()))
    return {"room_mIoU": miou_r, "icon_mIoU": miou_i, "room_pixacc": acc_r,
            "score": (miou_r + miou_i) / 2,
            "room_IoU": {c: round(float(v), 4) for c, v in zip(ROOM_CLASSES, iou_r)},
            "icon_IoU": {c: round(float(v), 4) for c, v in zip(ICON_CLASSES, iou_i)}}


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--probe", action="store_true")
    args = ap.parse_args()
    cfg = json.load(open(args.config))
    torch.backends.cudnn.benchmark = True

    if args.probe:
        probe(cfg)
        return

    distributed = "RANK" in os.environ
    if distributed:
        dist.init_process_group("nccl" if torch.cuda.is_available() else "gloo",
                                timeout=datetime.timedelta(minutes=60))
        rank, world = dist.get_rank(), dist.get_world_size()
        local = int(os.environ["LOCAL_RANK"])
    else:
        rank, world, local = 0, 1, 0
    if torch.cuda.is_available():
        dev = torch.device("cuda", local)
        torch.cuda.set_device(dev)
    else:  # CPU smoke-test path only
        dev = torch.device("cpu")
    is_main = rank == 0
    seed = cfg["seed"] + rank
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    def bcast(value):
        """Broadcast a float from rank 0 so every rank takes the same decision."""
        t = torch.tensor([float(value)], device=dev)
        if world > 1:
            dist.broadcast(t, 0)
        return t.item()

    # data
    train_ds = CachedFloorplans(cfg["cache_root"], "train", True, cfg["crop"], cfg["sigma"])
    val_ds = CachedFloorplans(cfg["cache_root"], "val", False, cfg["crop"], cfg["sigma"])
    sampler = torch.utils.data.distributed.DistributedSampler(
        train_ds, num_replicas=world, rank=rank, shuffle=True, seed=cfg["seed"], drop_last=True)
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=cfg["batch_per_gpu"], sampler=sampler, num_workers=cfg["workers"],
        pin_memory=dev.type == "cuda", drop_last=True, persistent_workers=True, prefetch_factor=4,
        worker_init_fn=seed_worker)
    spe = len(train_loader)

    # model (+ resume)
    os.makedirs(cfg["ckpt_dir"], exist_ok=True)
    last_path = os.path.join(cfg["ckpt_dir"], "last.pt")
    best_path = os.path.join(cfg["ckpt_dir"], "best.pt")
    ckpt = None
    if os.path.exists(last_path):
        ckpt = torch.load(last_path, map_location="cpu", weights_only=False)
        if ckpt.get("signature") != signature(cfg):
            raise RuntimeError(f"Checkpoint config {ckpt.get('signature')} != current "
                               f"{signature(cfg)}. Use a new HF repo name or matching settings.")
    pretrained = None if (ckpt or os.environ.get("FP_NO_PRETRAIN")) else "imagenet"
    model = build_model(cfg["encoder"], pretrained)
    if ckpt:
        model.load_state_dict(ckpt["model"])
    if world > 1 and dev.type == "cuda":
        from timm.layers import convert_sync_batchnorm
        model = convert_sync_batchnorm(model)
    model = model.to(dev)
    ema = EMA(model, cfg["ema_decay"])
    if ckpt:
        ema.module.load_state_dict(ckpt["ema"])
        ema.updates = ckpt.get("ema_updates", 0)
    ddp = (nn.parallel.DistributedDataParallel(
        model, device_ids=[local] if dev.type == "cuda" else None, gradient_as_bucket_view=True)
        if world > 1 else model)
    opt = make_optimizer(model, cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")
    crit = MultiTaskLoss(*cfg["loss_weights"])

    start_epoch, best_score, total_epochs, history = 0, -1.0, None, []
    if ckpt:
        opt.load_state_dict(ckpt["opt"])
        for g, bl in zip(opt.param_groups, ckpt["base_lrs"]):
            g["base_lr"] = bl
        scaler.load_state_dict(ckpt["scaler"])
        start_epoch, best_score = ckpt["epoch"], ckpt["best_score"]
        total_epochs, history = ckpt["total_epochs"], ckpt.get("history", [])
        log(f"Resumed at epoch {start_epoch} (best score {best_score:.4f}, "
            f"schedule {total_epochs} epochs)") if is_main else None
    del ckpt
    if cfg.get("force_total_epochs"):
        total_epochs = int(cfg["force_total_epochs"])
    adaptive = total_epochs is None
    if adaptive:
        total_epochs = cfg["max_epochs"]
    uploader = Uploader(cfg.get("hf_repo"), os.environ.get("HF_TOKEN")) if is_main else None
    deadline = cfg["deadline_ts"]
    if is_main:
        log(f"world={world} batch/gpu={cfg['batch_per_gpu']} steps/epoch={spe} "
            f"train={len(train_ds)} val={len(val_ds)} epochs {start_epoch}->{total_epochs}")

    epoch_secs = None
    stopped_by_time = False
    epoch = start_epoch
    for epoch in range(start_epoch, total_epochs):
        # ---- time guard (rank 0 decides)
        if epoch_secs is not None:
            need = epoch_secs * 1.15 + cfg["final_margin_s"]
            stop = bcast(1.0 if time.time() + need > deadline else 0.0)
            if stop:
                stopped_by_time = True
                if is_main:
                    log(f"Time budget reached before epoch {epoch}; stopping cleanly.")
                break
        t_ep = time.time()
        ddp.train()
        sampler.set_epoch(epoch)
        run = {"loss": 0.0, "room": 0.0, "icon": 0.0, "heat": 0.0}
        n_seen = 0
        for it, (x, room, icon, heat) in enumerate(train_loader):
            step = epoch * spe + it
            set_lr(opt, step, spe, total_epochs, cfg["warmup_epochs"])
            x = x.to(dev, non_blocking=True)
            room, icon = room.to(dev, non_blocking=True), icon.to(dev, non_blocking=True)
            heat = heat.to(dev, non_blocking=True)
            with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
                out = ddp(x)
            loss, parts = crit(out, room, icon, heat)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            ema.update(model)
            if torch.isfinite(loss):
                run["loss"] += loss.item(); n_seen += 1
                for k in parts:
                    run[k] += parts[k].item()
            if is_main and ((it + 1) % cfg["log_every"] == 0 or it == 0 or it + 1 == spe):
                d = max(1, n_seen)
                el = time.time() - t_ep
                ips = (it + 1) * cfg["batch_per_gpu"] * world / max(el, 1e-6)
                ep_eta = el / (it + 1) * (spe - it - 1)
                per_ep = epoch_secs or (el / (it + 1) * spe * 1.15)
                run_eta = ep_eta + per_ep * (total_epochs - epoch - 1)
                frac = (it + 1) / spe
                bar = "#" * int(frac * 20) + "-" * (20 - int(frac * 20))
                log(f"ep {epoch + 1}/{total_epochs} [{bar}] {frac * 100:5.1f}% it {it + 1}/{spe} | "
                    f"loss {run['loss'] / d:.3f} (room {run['room'] / d:.3f} icon {run['icon'] / d:.3f} "
                    f"heat {run['heat'] / d:.3f}) | {ips:.1f} img/s | epoch ETA {_hms(ep_eta)} | "
                    f"run ETA {_hms(run_eta)} | best {max(best_score, 0):.4f}")
        train_secs = time.time() - t_ep

        # ---- validation (EMA weights)
        metrics = None
        if (epoch + 1) % cfg["val_every"] == 0 or epoch + 1 == total_epochs:
            if is_main:
                log(f"ep {epoch + 1}: training done in {_hms(time.time() - t_ep)}, validating {len(val_ds)} plans ...")
            metrics = validate(ema.module, val_ds, rank, world, dev)
        epoch_secs = time.time() - t_ep

        # ---- adapt the cosine schedule to the time budget (fresh runs, after epoch 0)
        if adaptive and epoch == start_epoch:
            remaining = deadline - time.time() - cfg["final_margin_s"]
            fit = epoch + 1 + int(remaining // (epoch_secs * 1.08))
            new_total = int(bcast(max(epoch + 2, min(cfg["max_epochs"], fit))))
            if is_main:
                log(f"Epoch takes {epoch_secs / 60:.1f} min -> schedule set to {new_total} epochs "
                    f"to finish inside the time budget.")
            total_epochs, adaptive = new_total, False

        # ---- checkpoint + upload (rank 0)
        if is_main:
            d = max(1, n_seen)
            rec = {"epoch": epoch, "train_loss": run["loss"] / d, "room_loss": run["room"] / d,
                   "icon_loss": run["icon"] / d, "heat_loss": run["heat"] / d,
                   "epoch_min": round(epoch_secs / 60, 2), "train_min": round(train_secs / 60, 2)}
            improved = False
            if metrics:
                rec.update(metrics)
                improved = metrics["score"] > best_score
                if improved:
                    best_score = metrics["score"]
                log(f"VAL ep {epoch}: room mIoU {metrics['room_mIoU']:.4f} icon mIoU "
                    f"{metrics['icon_mIoU']:.4f} | wall {metrics['room_IoU']['Wall']:.3f} "
                    f"door {metrics['icon_IoU']['Door']:.3f} window {metrics['icon_IoU']['Window']:.3f} "
                    f"stairs {metrics['icon_IoU']['Stairs']:.3f} {'*BEST*' if improved else ''}")
            history.append(rec)
            state = {"model": model.state_dict() if world == 1 else _unsync_state(model),
                     "ema": ema.module.state_dict(), "ema_updates": ema.updates,
                     "opt": opt.state_dict(), "base_lrs": [g["base_lr"] for g in opt.param_groups],
                     "scaler": scaler.state_dict(), "epoch": epoch + 1, "best_score": best_score,
                     "total_epochs": total_epochs, "history": history,
                     "signature": signature(cfg), "config": cfg | {"hf_token": None}}
            atomic_save(state, last_path)
            files = [(last_path, "checkpoints/last.pt")]
            if improved:
                atomic_save({"ema": ema.module.state_dict(), "epoch": epoch, "metrics": metrics,
                             "signature": signature(cfg), "room_classes": ROOM_CLASSES,
                             "icon_classes": ICON_CLASSES, "encoder": cfg["encoder"]}, best_path)
                files.append((best_path, "checkpoints/best.pt"))
            mpath = os.path.join(cfg["ckpt_dir"], "metrics.json")
            json.dump(history, open(mpath, "w"), indent=1)
            files.append((mpath, "metrics.json"))
            uploader.submit(files, f"epoch {epoch}" + (" (best)" if improved else ""))
        if world > 1:
            dist.barrier()

    # ---- finish
    if is_main:
        done = (not stopped_by_time) and len(history) > 0 and history[-1]["epoch"] + 1 >= total_epochs
        status = {"finished_schedule": bool(done), "stopped_by_time": stopped_by_time,
                  "epochs_done": (history[-1]["epoch"] + 1) if history else start_epoch,
                  "total_epochs": total_epochs, "best_score": best_score,
                  "time": time.strftime("%Y-%m-%d %H:%M:%S")}
        spath = os.path.join(cfg["ckpt_dir"], "status.json")
        json.dump(status, open(spath, "w"), indent=1)
        uploader.submit([(spath, "status.json")], "run status")
        uploader.wait(timeout=cfg["final_margin_s"] - 60)
        log("STATUS", json.dumps(status))
    if world > 1:
        dist.barrier()
        dist.destroy_process_group()


def _hms(sec):
    sec = int(max(0, sec))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m" if sec >= 3600 else f"{sec // 60}m{sec % 60:02d}s"


def _unsync_state(model):
    # SyncBatchNorm and BatchNorm share parameter/buffer names, so the state dict loads
    # straight into the plain (non-sync) model on resume / inference.
    return model.state_dict()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/fp_src/viz.py
"""Render best-checkpoint predictions on a few validation plans and upload the image to HF."""
import json
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch

from fp_common import CachedFloorplans, build_model, split_logits, MEAN, STD, ROOM_CLASSES, ICON_CLASSES


def main(cfg_path, n=4):
    cfg = json.load(open(cfg_path))
    best = os.path.join(cfg["ckpt_dir"], "best.pt")
    if not os.path.exists(best):
        print("No best.pt yet; skipping visualisation.")
        return
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    ck = torch.load(best, map_location="cpu", weights_only=False)
    net = build_model(cfg["encoder"], None)
    net.load_state_dict(ck["ema"])
    net.to(dev).eval()
    ds = CachedFloorplans(cfg["cache_root"], "val", False, cfg["crop"], cfg["sigma"])
    idx = np.linspace(0, len(ds) - 1, n).astype(int)
    cmap_r = plt.get_cmap("tab20", len(ROOM_CLASSES))
    cmap_i = plt.get_cmap("tab20", len(ICON_CLASSES))
    fig, axes = plt.subplots(n, 5, figsize=(25, 5 * n))
    axes = np.atleast_2d(axes)
    for row, i in enumerate(idx):
        x, room, icon, heat = ds[int(i)]
        with torch.no_grad(), torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
            out = net(x[None].to(dev)).float()
        r, ic, h = split_logits(out)
        img = (x.numpy().transpose(1, 2, 0) * STD + MEAN).clip(0, 255).astype(np.uint8)
        panels = [(img, None, "input"),
                  (room.numpy(), cmap_r, "rooms GT"), (r.argmax(1)[0].cpu().numpy(), cmap_r, "rooms pred"),
                  (ic.argmax(1)[0].cpu().numpy(), cmap_i, "icons pred (door/window/stairs...)"),
                  (torch.sigmoid(h)[0].max(0).values.cpu().numpy(), "magma", "junction heatmaps")]
        for ax, (im, cm, title) in zip(axes[row], panels):
            if cm is None:
                ax.imshow(im)
            elif isinstance(cm, str):
                ax.imshow(im, cmap=cm, vmin=0, vmax=1)
            else:
                ax.imshow(np.where(im == 255, 0, im), cmap=cm, vmin=0, vmax=cm.N - 1,
                          interpolation="nearest")
            ax.set_title(title)
            ax.axis("off")
    plt.tight_layout()
    out_png = os.path.join(cfg["ckpt_dir"], "val_predictions.png")
    plt.savefig(out_png, dpi=60)
    print("Saved", out_png, "| best epoch", ck["epoch"], "| score", round(ck["metrics"]["score"], 4))
    tok = os.environ.get("HF_TOKEN")
    if tok and cfg.get("hf_repo"):
        try:
            from huggingface_hub import HfApi
            HfApi(token=tok).upload_file(path_or_fileobj=out_png, path_in_repo="val_predictions.png",
                                         repo_id=cfg["hf_repo"], commit_message="validation samples")
            print("Uploaded val_predictions.png")
        except Exception as e:
            print("Upload of preview failed:", e)


if __name__ == "__main__":
    main(sys.argv[1])


In [ ]:
%%writefile /kaggle/working/fp_src/ddp_check.py
"""Quick 2-GPU communication check (all_reduce + barrier)."""
import datetime
import os

import torch
import torch.distributed as dist

dist.init_process_group("nccl", timeout=datetime.timedelta(seconds=90))
r = dist.get_rank()
torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
t = torch.ones(1024, 1024, device="cuda") * (r + 1)
for _ in range(5):
    dist.all_reduce(t)
dist.barrier()
torch.cuda.synchronize()
if r == 0:
    print("DDP_OK", float(t[0, 0]), flush=True)
dist.destroy_process_group()


In [ ]:
# ===================== SANITY CHECKS (fail fast, clear messages) =====================
import urllib.request
def fail(msg): raise RuntimeError("\n\n*** SETUP PROBLEM: " + msg + " ***\n")

# GPUs
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
print("\n".join(gpus))
if len(gpus) < 2: fail("Need 2 GPUs. Settings -> Accelerator -> 'GPU T4 x2'.")

# Internet
try: urllib.request.urlopen("https://huggingface.co", timeout=20)
except Exception as e: fail(f"No internet ({e}). Settings -> Internet -> On.")

# HF token (Kaggle Secret)
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Kaggle secret not readable:", e)
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if not HF_TOKEN: fail("Add a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets) and attach it to this notebook.")
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
try: HF_USER = api.whoami()["name"]
except Exception as e: fail(f"HF token rejected: {e}")
MODEL_REPO, DATASET_REPO = f"{HF_USER}/{MODEL_REPO_NAME}", f"{HF_USER}/{DATASET_REPO_NAME}"
try:
    api.create_repo(MODEL_REPO, private=True, exist_ok=True)
    api.create_repo(DATASET_REPO, repo_type="dataset", private=True, exist_ok=True)
except Exception as e: fail(f"Token cannot create repos (needs WRITE access): {e}")

readme = f"""---
license: cc-by-nc-sa-4.0
tags: [floorplan, segmentation, multitask, cubicasa5k]
---
# Floorplan multi-task model ({ENCODER} U-Net, trained at {CROP}px)
Heads (27 output channels): rooms 0-11, icons 12-23, junction heatmaps 24-26 (sigmoid).
* Rooms: Background, Outdoor, Wall, Kitchen, LivingRoom, Bedroom, Bath, Entry/Hallway, Railing, Storage, Garage, OtherRoom
* Icons: NoIcon, Window, Door, Closet, ElectricalAppliance, Toilet, Sink, SaunaBench, Fireplace, Bathtub, Chimney, Stairs
* Heatmaps: WallJunction, OpeningEndpoint, IconCorner
Input: RGB, ImageNet mean/std normalisation, sides padded to a multiple of 32 (white).
Load: `smp.Unet("{ENCODER}", encoder_weights=None, classes=27, decoder_attention_type="scse")`,
then `load_state_dict(torch.load("checkpoints/best.pt")["ema"])`.
Progress: `metrics.json` (per epoch) and `status.json`. Trained on CubiCasa5K (CC BY-NC-SA 4.0).
"""
try:
    api.upload_file(path_or_fileobj=readme.encode(), path_in_repo="README.md", repo_id=MODEL_REPO,
                    commit_message="model card")
except Exception as e: fail(f"Write test to {MODEL_REPO} failed: {e}")
print("HF OK ->", f"https://huggingface.co/{MODEL_REPO}")

# Disk
free_gb = shutil.disk_usage(BIG_DIR).free / 1e9
print(f"free disk in {BIG_DIR}: {free_gb:.0f} GB")
if free_gb < 14: fail(f"Only {free_gb:.0f} GB free in {BIG_DIR}; need ~14 GB.")

# Pretrained encoder weights (download once, before the 2 workers start)
r = subprocess.run([sys.executable, "-c",
    "import timm; timm.create_model('efficientnet_b3', pretrained=True); print('weights ok')"],
    capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-800:])
if r.returncode != 0: fail("Could not download EfficientNet-B3 ImageNet weights.")

# 2-GPU communication test (falls back to NCCL_P2P_DISABLE=1 if needed)
DDP_ENV = dict(os.environ, OMP_NUM_THREADS="2", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
def ddp_ok(env):
    try:
        r = subprocess.run(["torchrun", "--standalone", "--nproc_per_node=2", f"{SRC_DIR}/ddp_check.py"],
                           capture_output=True, text=True, timeout=240, env=env)
        return "DDP_OK" in r.stdout, (r.stdout + r.stderr)[-1500:]
    except subprocess.TimeoutExpired:
        return False, "timeout"
ok, out = ddp_ok(DDP_ENV)
if not ok:
    print("Default NCCL failed, retrying with NCCL_P2P_DISABLE=1 ...\n", out)
    DDP_ENV["NCCL_P2P_DISABLE"] = "1"
    ok, out = ddp_ok(DDP_ENV)
if not ok: fail("2-GPU NCCL test failed:\n" + out)
print("2-GPU DDP OK", "(P2P disabled)" if DDP_ENV.get("NCCL_P2P_DISABLE") else "")
print(f"setup done in {(time.time()-NOTEBOOK_START)/60:.1f} min")

In [ ]:
# ===================== DATA (HF cache -> Kaggle input -> Zenodo) =====================
sys.path.insert(0, SRC_DIR)
import fp_prep
t0 = time.time()
index = fp_prep.prepare_data(CACHE_DIR, f"{BIG_DIR}/work", f"{BIG_DIR}/parser", MAX_SIDE,
                             api, DATASET_REPO, HF_TOKEN)
print({k: len(v) for k, v in index.items() if k != "meta"}, "| meta:",
      {k: v for k, v in index["meta"].items() if k != "failures"},
      "| failed:", len(index["meta"]["failures"]))
print(f"data ready in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ===================== RESUME (pull last checkpoint from HF if present) =====================
from huggingface_hub import hf_hub_download
remote = set(api.list_repo_files(MODEL_REPO))
for f in ["checkpoints/last.pt", "checkpoints/best.pt"]:
    local = os.path.join(CKPT_DIR, os.path.basename(f))
    if f in remote and not os.path.exists(local):
        for attempt in range(3):
            try:
                p = hf_hub_download(MODEL_REPO, f, token=HF_TOKEN, local_dir=f"{BIG_DIR}/hf_ckpt")
                shutil.copyfile(p, local); print("restored", f); break
            except Exception as e:
                print("download retry", attempt + 1, e); time.sleep(20)
        else:
            raise RuntimeError(f"Could not download {f}; rerun later rather than restarting from scratch.")
force_total = None
if os.path.exists(os.path.join(CKPT_DIR, "last.pt")):
    import torch
    ck = torch.load(os.path.join(CKPT_DIR, "last.pt"), map_location="cpu", weights_only=False)
    print(f"resuming: epoch {ck['epoch']}/{ck['total_epochs']}, best score {ck['best_score']:.4f}")
    if EXTRA_EPOCHS > 0:
        force_total = ck["total_epochs"] + EXTRA_EPOCHS
    elif ck["epoch"] >= ck["total_epochs"]:
        print("Schedule already finished. Set EXTRA_EPOCHS > 0 to keep training.")
    del ck
else:
    print("fresh run")

In [ ]:
# ===================== RUN CONFIG + BATCH-SIZE PROBE =====================
cpus = os.cpu_count() or 4
cfg = {"cache_root": CACHE_DIR, "ckpt_dir": CKPT_DIR, "encoder": ENCODER, "crop": CROP,
       "max_side": MAX_SIDE, "sigma": SIGMA, "batch_per_gpu": 4, "workers": max(2, min(4, cpus // 2)),
       "lr": LR, "encoder_lr_mult": 0.5, "weight_decay": 1e-2, "warmup_epochs": 1,
       "max_epochs": MAX_EPOCHS, "force_total_epochs": force_total, "ema_decay": 0.999,
       "grad_clip": 1.0, "loss_weights": [1.0, 1.0, 1.0], "val_every": 1, "log_every": 50,
       "seed": SEED, "final_margin_s": 900, "probe_batches": [8, 6, 5, 4, 3, 2],
       "deadline_ts": NOTEBOOK_START + TIME_BUDGET_HOURS * 3600, "hf_repo": MODEL_REPO}
CFG_PATH = f"{SRC_DIR}/run_config.json"
json.dump(cfg, open(CFG_PATH, "w"), indent=1)
r = subprocess.run([sys.executable, f"{SRC_DIR}/train.py", "--config", CFG_PATH, "--probe"],
                   capture_output=True, text=True, cwd=SRC_DIR, env=DDP_ENV)
print("\n".join(l for l in r.stdout.splitlines() if "probe" in l.lower()))
m = re.search(r"PROBE_BS=(\d+)", r.stdout)
if not m: print(r.stderr[-2000:]); raise RuntimeError("Batch probe failed (see error above).")
cfg["batch_per_gpu"] = int(m.group(1))
json.dump(cfg, open(CFG_PATH, "w"), indent=1)
print(f"batch per GPU = {cfg['batch_per_gpu']} (global {2 * cfg['batch_per_gpu']}), workers/GPU = {cfg['workers']}")

In [ ]:
# ===================== TRAIN on both T4s (auto-restarts from last.pt on a crash) =====================
def launch():
    cmd = ["torchrun", "--standalone", "--nproc_per_node=2", "train.py", "--config", CFG_PATH]
    p = subprocess.Popen(cmd, cwd=SRC_DIR, env=DDP_ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Warning" in line or "warnings.warn" in line: continue
        print(line, end="", flush=True)
    return p.wait()

for attempt in range(1, 4):
    rc = launch()
    status_p = os.path.join(CKPT_DIR, "status.json")
    if rc == 0 and os.path.exists(status_p):
        print("\nTraining exited cleanly:", open(status_p).read()); break
    left_h = (cfg["deadline_ts"] - time.time()) / 3600
    print(f"\n!!! training exited with code {rc} (attempt {attempt}); {left_h:.1f} h left in budget")
    if left_h < 0.75: break
    cfg["force_total_epochs"] = None   # keep the schedule stored in last.pt
    json.dump(cfg, open(CFG_PATH, "w"), indent=1)
    time.sleep(30)

In [ ]:
# ===================== PREVIEW + SUMMARY =====================
subprocess.run([sys.executable, f"{SRC_DIR}/viz.py", CFG_PATH], cwd=SRC_DIR, env=DDP_ENV)
from IPython.display import Image, display
png = os.path.join(CKPT_DIR, "val_predictions.png")
if os.path.exists(png): display(Image(png))
hist = json.load(open(os.path.join(CKPT_DIR, "metrics.json"))) if os.path.exists(os.path.join(CKPT_DIR, "metrics.json")) else []
if hist:
    b = max((h for h in hist if "score" in h), key=lambda h: h["score"])
    print(f"best epoch {b['epoch']}: room mIoU {b['room_mIoU']:.4f}  icon mIoU {b['icon_mIoU']:.4f}")
    print("room IoU:", b["room_IoU"]); print("icon IoU:", b["icon_IoU"])
print(f"total runtime {(time.time()-NOTEBOOK_START)/3600:.2f} h  ->  https://huggingface.co/{MODEL_REPO}")